# Traditional ML subtype models - ACCURACY optimized

For each subtype, the ACCURACY-optimized configuration is selected by cross-validation on the official training set. The testing set is loaded only after all choices are frozen.


In [ ]:
from pathlib import Path
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OBJECTIVE = "accuracy"
SUBTYPES = ["Credibility and Obstinacy", "Compliance", "Descriptors", "Misgendering"]
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
official_train = pd.read_csv(DATA_DIR / "GEP_train_80_20.csv")
train_texts = official_train["text"].astype(str).tolist()

vectorizers = {
    "tfidf": TfidfVectorizer(lowercase=True),
    "count": CountVectorizer(lowercase=True),
    "tfidf_sw": TfidfVectorizer(lowercase=True, stop_words="english"),
    "count_sw": CountVectorizer(lowercase=True, stop_words="english"),
}
families = {
    "SVM": (LinearSVC(class_weight="balanced", max_iter=5000), {"clf__C": [0.01, 0.1, 1, 10]}),
    "RF": (RandomForestClassifier(class_weight="balanced", random_state=42), {"clf__n_estimators": [100, 300], "clf__max_depth": [None, 20, 40]}),
    "LR": (LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=1000), {"clf__C": [0.01, 0.1, 1, 10]}),
    "NB": (MultinomialNB(), {"clf__alpha": [0.1, 1.0, 10.0]}),
}
vectorizer_grid = {
    "vectorizer__max_features": [10000, 20000],
    "vectorizer__ngram_range": [(1, 1), (1, 2)],
    "vectorizer__min_df": [3, 5],
}

# Select one configuration per subtype using training-set cross-validation only.
selected = {}
for subtype in SUBTYPES:
    labels = official_train[subtype].astype(int).to_numpy()
    candidates = {}
    for family, (classifier, classifier_grid) in families.items():
        for vectorizer_name, vectorizer in vectorizers.items():
            name = f"{family}_{vectorizer_name}"
            pipeline = Pipeline([("vectorizer", vectorizer), ("clf", classifier)])
            search = GridSearchCV(
                pipeline, {**vectorizer_grid, **classifier_grid}, cv=CV,
                scoring=OBJECTIVE, n_jobs=-1, refit=True,
            )
            search.fit(train_texts, labels)
            candidates[name] = search
    selected[subtype] = max(candidates.items(), key=lambda item: item[1].best_score_)

model_output = MODEL_DIR / "subtypes_traditional_accuracy"
model_output.mkdir(parents=True, exist_ok=True)
for subtype, (name, search) in selected.items():
    safe_subtype = subtype.lower().replace(" ", "_")
    joblib.dump(search.best_estimator_, model_output / f"{safe_subtype}__{name}.joblib")

# The held-out testing set is opened only after all subtype configurations are frozen.
heldout_test = pd.read_csv(DATA_DIR / "GEP_test_80_20.csv")
test_texts = heldout_test["text"].astype(str).tolist()
rows = []
for subtype, (name, search) in selected.items():
    labels = heldout_test[subtype].astype(int).to_numpy()
    predictions = search.best_estimator_.predict(test_texts)
    if hasattr(search.best_estimator_, "predict_proba"):
        scores = search.best_estimator_.predict_proba(test_texts)[:, 1]
    else:
        scores = search.best_estimator_.decision_function(test_texts)
    rows.append({
        "subtype": subtype,
        "configuration": name,
        "selection_objective": OBJECTIVE,
        "training_cv_score": search.best_score_,
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, zero_division=0),
        "roc_auc": roc_auc_score(labels, scores) if len(np.unique(labels)) == 2 else np.nan,
        "best_parameters": search.best_params_,
    })

heldout_results = pd.DataFrame(rows)
heldout_results.to_csv(RESULTS_DIR / "subtype_traditional_accuracy_heldout_results.csv", index=False)
heldout_results
